# Analyzing CliMA development speed

How quickly does CliMA's ecosystem turn pull requests around? Here we measure
each merged PR's *time-to-merge* (the wall-clock time from when it was opened to
when it was merged) and track the **median time-to-merge per quarter** over
time, both per repository and across the whole ecosystem.

Bot-authored PRs (CompatHelper, TagBot, Dependabot, GitHub Actions, etc.) are
excluded so the metric reflects human development speed rather than automated
dependency bumps.


In [1]:
%cd ..

/Users/pete/calkit/clima-perf


## Load merged pull requests

`fetch-github` stores merged PRs under `data/github/prs/<repo>/<YYYY-MM-DD>.json`
(one file per UTC day a PR was merged in that repo). Each record carries
`createdAt` and `mergedAt`, from which we derive the time-to-merge.


In [2]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import plotly.colors as pc
import plotly.graph_objects as go

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT / "clima-perf" / "data").exists():
    ROOT = ROOT / "clima-perf"
DATA = ROOT / "data"
PRS_ROOT = DATA / "github" / "prs"

# Anything before 2018-Q1 predates the CliMA packaging effort; the time
# axis is cropped to that floor (matches the analyze notebook).
MIN_QUARTER = "2018-Q1"

# Minimum number of merged PRs in a repo-quarter for that point to be
# plotted; medians over only one or two PRs are too noisy to be useful.
MIN_PRS_PER_POINT = 3

# Bot accounts whose ``is_bot`` field is false on the GitHub API but which
# are clearly automated (Julia ecosystem bots).
EXTRA_BOTS = {"JuliaTagBot", "attobot"}


def is_bot(author: dict) -> bool:
    login = author.get("login") or ""
    return bool(
        author.get("is_bot")
        or login in EXTRA_BOTS
        or login.endswith("[bot]")
        or login.startswith("app/")
    )


# CamelCase repo lookup so legend labels match the canonical package names.
camel_by_lc: dict[str, str] = {}
creation_path = DATA / "github" / "repo_creation_dates.jsonl"
if creation_path.exists():
    for line in creation_path.read_text().splitlines():
        if line.strip():
            name = json.loads(line)["name"].replace(".jl", "")
            camel_by_lc[name.lower()] = name


def camel(name: str) -> str:
    return camel_by_lc.get(name.lower(), name)


def parse_iso(s: str) -> datetime:
    dt = datetime.fromisoformat(s.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def quarter_of(dt: datetime) -> str:
    return f"{dt.year}-Q{(dt.month - 1) // 3 + 1}"


def quarter_start(quarter: str) -> datetime:
    year, q = quarter.split("-Q")
    return datetime(int(year), (int(q) - 1) * 3 + 1, 1, tzinfo=timezone.utc)


rows: list[dict] = []
if PRS_ROOT.exists():
    for repo_dir in sorted(PRS_ROOT.iterdir()):
        if not repo_dir.is_dir():
            continue
        repo = camel(repo_dir.name)
        for f in repo_dir.glob("*.json"):
            for pr in json.loads(f.read_text()):
                author = pr.get("author") or {}
                if is_bot(author):
                    continue
                created_raw = pr.get("createdAt")
                merged_raw = pr.get("mergedAt")
                if not created_raw or not merged_raw:
                    continue
                created = parse_iso(created_raw)
                merged = parse_iso(merged_raw)
                ttm_days = (merged - created).total_seconds() / 86400.0
                if ttm_days < 0:
                    continue
                quarter = quarter_of(merged)
                if quarter < MIN_QUARTER:
                    continue
                rows.append(
                    {
                        "repo": repo,
                        "number": pr.get("number"),
                        "author": author.get("login"),
                        "created_at": created_raw,
                        "merged_at": merged_raw,
                        "quarter": quarter,
                        "ttm_days": ttm_days,
                    }
                )

prs_df = pd.DataFrame(rows)
print(
    f"{len(prs_df)} merged non-bot PRs across "
    f"{prs_df['repo'].nunique() if not prs_df.empty else 0} repos"
)
prs_df.head()


11904 merged non-bot PRs across 22 repos


,repo,number,author,created_at,merged_at,quarter,ttm_days
0,CalibrateEmulateSample,85,jakebolewski,2020-11-18T00:57:16Z,2020-11-18T01:10:02Z,2020-Q4,0.008866
1,CalibrateEmulateSample,83,agarbuno,2020-11-03T03:29:19Z,2020-11-18T17:14:16Z,2020-Q4,15.572882
2,CalibrateEmulateSample,223,odunbar,2023-07-12T17:28:38Z,2023-07-12T19:17:23Z,2023-Q3,0.075521
3,CalibrateEmulateSample,253,bielim,2023-11-13T16:00:22Z,2023-12-11T22:37:22Z,2023-Q4,28.275694
4,CalibrateEmulateSample,246,odunbar,2023-11-01T19:51:15Z,2023-11-01T20:49:48Z,2023-Q4,0.040660


## Median time-to-merge per quarter

We bin merged PRs by the quarter they were merged in and take the median
time-to-merge. One trace is drawn per repository (only quarters with at least
`MIN_PRS_PER_POINT` merged PRs are plotted), and a thick gray line shows the
median computed over *all* repositories' PRs combined.


In [3]:
empty = prs_df.empty

# Per-repo, per-quarter median time-to-merge + PR count.
if empty:
    repo_q = pd.DataFrame(
        columns=["repo", "quarter", "n_prs", "median_ttm_days"]
    )
else:
    repo_q = (
        prs_df.groupby(["repo", "quarter"])["ttm_days"]
        .agg(n_prs="size", median_ttm_days="median")
        .reset_index()
    )

# Ecosystem-wide median over all repos, per quarter.
if empty:
    all_q = pd.DataFrame(columns=["repo", "quarter", "n_prs", "median_ttm_days"])
else:
    all_q = (
        prs_df.groupby("quarter")["ttm_days"]
        .agg(n_prs="size", median_ttm_days="median")
        .reset_index()
    )
    all_q.insert(0, "repo", "All repos")

ttm_df = pd.concat([repo_q, all_q], ignore_index=True)
ttm_df["quarter_start"] = ttm_df["quarter"].map(
    lambda q: quarter_start(q).date().isoformat()
)
ttm_df = ttm_df[
    ["repo", "quarter", "quarter_start", "n_prs", "median_ttm_days"]
].sort_values(["repo", "quarter"]).reset_index(drop=True)

os.makedirs(ROOT / "results", exist_ok=True)
ttm_df.to_csv(ROOT / "results/pr-ttm.csv", index=False)
print(f"wrote {len(ttm_df)} rows to results/pr-ttm.csv")
ttm_df.head()


wrote 424 rows to results/pr-ttm.csv


,repo,quarter,quarter_start,n_prs,median_ttm_days
0,All repos,2018-Q4,2018-10-01,5,0.004931
1,All repos,2019-Q1,2019-01-01,27,0.170949
2,All repos,2019-Q2,2019-04-01,71,0.139456
3,All repos,2019-Q3,2019-07-01,115,0.173380
4,All repos,2019-Q4,2019-10-01,181,0.104514


In [4]:
# Repos ordered by total merged-PR volume (busiest first) for a stable,
# meaningful legend order.
if empty:
    repo_order: list[str] = []
else:
    repo_order = (
        prs_df["repo"].value_counts().index.tolist()
    )

palette = (
    list(pc.qualitative.Plotly)
    + list(pc.qualitative.D3)
    + list(pc.qualitative.Light24)
    + list(pc.qualitative.Dark24)
)
color_for = {
    repo: palette[i % len(palette)] for i, repo in enumerate(repo_order)
}

fig = go.Figure()

for repo in repo_order:
    sub = repo_q[
        (repo_q["repo"] == repo) & (repo_q["n_prs"] >= MIN_PRS_PER_POINT)
    ].sort_values("quarter")
    if sub.empty:
        continue
    x = [quarter_start(q) for q in sub["quarter"]]
    fig.add_trace(
        go.Scatter(
            x=x,
            y=sub["median_ttm_days"],
            mode="lines+markers",
            name=repo,
            line=dict(color=color_for[repo], width=1.5),
            marker=dict(size=4),
            customdata=sub[["quarter", "n_prs"]],
            hovertemplate=(
                f"<b>{repo}</b><br>"
                "%{customdata[0]}<br>"
                "Median TTM: %{y:.1f} days<br>"
                "Merged PRs: %{customdata[1]}<extra></extra>"
            ),
        )
    )

# Thick gray ecosystem-wide line, drawn last so it sits on top.
if not all_q.empty:
    all_sorted = all_q.sort_values("quarter")
    fig.add_trace(
        go.Scatter(
            x=[quarter_start(q) for q in all_sorted["quarter"]],
            y=all_sorted["median_ttm_days"],
            mode="lines",
            name="All repos",
            line=dict(color="#666666", width=4),
            customdata=all_sorted[["quarter", "n_prs"]],
            hovertemplate=(
                "<b>All repos</b><br>"
                "%{customdata[0]}<br>"
                "Median TTM: %{y:.1f} days<br>"
                "Merged PRs: %{customdata[1]}<extra></extra>"
            ),
        )
    )

fig.update_layout(
    title="Pull request time-to-merge",
    xaxis_title="Quarter",
    yaxis_title="Median time-to-merge (days)",
    height=600,
    legend=dict(itemsizing="constant", font=dict(size=11)),
)

os.makedirs(ROOT / "figures", exist_ok=True)
fig.write_json(ROOT / "figures/pr-ttm.json")
fig
